# iPhone: pipeline dengan satu paket bobot per encoder

Validation lengkap terbaru mencakup 32 lagu / 451 section. Kandidat menghasilkan label identik dengan pipeline sebelumnya: accuracy 79,1574%, macro F1 0,767214 dan seluruh F1 per kelas tetap. Compiled encoder turun dari 1009,33 MiB menjadi 351,91 MiB (hemat 65,13%). Ini ukuran dua encoder, bukan keseluruhan aplikasi. Tidak ada klaim speedup dari ukuran file.

Probe terpisah ini membawa **satu paket multi-function MusicFM dan satu MuQ**. Fungsi `short_30`, `long_120` dan `partial` dipilih eksplisit dari masing-masing paket. Tidak membawa lagi paket encoder statis/partial terpisah. Log-mel dan convolution frontend tetap CPU, partial encoder tetap CPU, dan head/konteks epoch 5 tetap. Mode CPU+GPU hanya untuk full window dari graph yang diperbaiki.

Persiapan memverifikasi report, snapshot, cache per lagu, sumber dan hash model, menjalankan typecheck Swift dengan SDK iPhone, serta menjalankan ulang dua PCM pipeline di Mac terhadap hasil canonical yang sama. Strict feature/probability/label gates tetap; tidak memasang model ke SongMap dan tidak memakai data test. Proyek baru selalu dibuat di `outputs.local`.


In [ ]:
from pathlib import Path
from datetime import datetime
import hashlib,json,subprocess
ROOT=Path.cwd().resolve()
assert (ROOT/'scripts/prepare_canonical_pipeline_iphone.py').is_file(), 'Buka dari root SongMap'
SONGSTRUCT_ROOT=ROOT.parent/'Section-Annotations'
RUN_PYTHON=SONGSTRUCT_ROOT/'.venv-songformer311/bin/python'
assert RUN_PYTHON.is_file(), f'Python venv tidak ditemukan: {RUN_PYTHON}'
def sha(path):return hashlib.sha256(Path(path).read_bytes()).hexdigest()
EXPECTED_QUALITY='020f8d9a2bb5c29318f591a7c58d30b8f4cdc39eba47cf82c09820b538ac96fa'
QUALITY=next((p.resolve() for p in (ROOT/'outputs.local').glob('encoder-canonical-bank-*/quality/validation-all.json') if sha(p)==EXPECTED_QUALITY),None)
assert QUALITY, 'Report canonical full validation yang sudah ditinjau belum ditemukan'
q=json.loads(QUALITY.read_text())
assert q['selection_identity']=='d0aba90759d73e74dbc629950916531b90c6dfe219ac5d1f355667cc686944bf'
OUTPUT=ROOT/'outputs.local'/('canonical-pipeline-iphone-'+datetime.now().strftime('%Y%m%d-%H%M%S-%f'))
print('Quality:',QUALITY)
print('Proyek baru:',OUTPUT)

def run_logged(args,log):
    log=Path(log);log.parent.mkdir(parents=True,exist_ok=True)
    with log.open('w',encoding='utf-8') as handle:
        process=subprocess.Popen([str(v) for v in args],cwd=ROOT,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        try:
            for line in process.stdout:
                print(line,end='');handle.write(line);handle.flush()
            status=process.wait()
        except KeyboardInterrupt:
            process.terminate();process.wait();raise
    if status:raise RuntimeError(f'Persiapan gagal ({status}); lihat {log}. Simpan mac-preflight.json bila ada; gunakan OUTPUT baru untuk percobaan ulang.')


In [ ]:
run_logged([RUN_PYTHON,'-u',ROOT/'scripts/prepare_canonical_pipeline_iphone.py',
    '--songstruct-root',SONGSTRUCT_ROOT,'--songmap-root',ROOT,
    '--quality-report',QUALITY,'--output-dir',OUTPUT],Path(str(OUTPUT)+'.prepare.log'))
proof=json.loads((OUTPUT/'preparation.json').read_text())
assert proof['native_typecheck_passed'] and proof['mac_pcm_pipeline_passed']
assert proof['quality_sha256']==EXPECTED_QUALITY and proof['canonical_encoder_policy']=='long-profile-shared-palette-v1'
print('Compiled encoder MiB:',proof['compiled_encoder_bytes']/2**20)
print('Seluruh model probe MiB:',proof['temporary_model_bytes']/2**20)
for check in proof['checks']:print(check['track_id'],check['native_context'])
subprocess.run(['open',str(OUTPUT/'Benchmark.xcodeproj')],check=True)


Di Xcode pilih Team signing dan iPhone fisik, lalu Run aplikasi **Canonical Pipeline**. Tidak perlu launch argument SongMap.

1. Pilih lagu pertama (Alleluia), mode **CPU only**, lalu **Run selected track**. Periksa `label_parity_passed=true`.
2. Jika lolos, pilih **CPU + GPU candidate** untuk lagu yang sama. Jalankan dan bagikan kedua JSON melalui **Share result JSON**. Waktu, loading, fungsi yang dipilih, requested compute units, thermal state dan sampled memory dicatat.
3. Jika keduanya lolos dan tidak force close, uji lagu panjang kedua. Setelah pilot lolos, jalankan **Run 3 passes over all tracks** pada CPU+GPU untuk enam run dalam satu proses dan bagikan batch JSON. Batch berhenti bila label/probability parity gagal.

Jika crash, buka kembali aplikasi dan bagikan **Share last recorded stage** sebelum menimpa checkpoint. Native assertion tidak bisa selalu ditangkap Swift; tidak ada fallback CPU diam-diam. Requested compute units tidak membuktikan hardware placement. Sampled footprint bukan peak memory, dan pengukuran dimulai dari bundled PCM sehingga MP3 decoder/MusicUnderstanding/chord/lyric tidak tercakup. Perbandingan dengan probe lama bukan benchmark AB/BA dalam kondisi cache identik.

Kandidat belum dipasang ke aplikasi utama. Menjaga hasil di validation tidak menjamin tidak ada regresi pada lagu lain; test split tidak digunakan pada tahap ini.
